# Lab Day 2 — DeepWeeds: backbone, công thức huấn luyện, suy luận (Kaggle / Colab)

Notebook tự nhận biết đang chạy trên Kaggle hay Colab.

**Kaggle (khuyên dùng, chạy nền được):** *Settings*: Accelerator = **GPU T4 x2** (hoặc P100), **Internet = On** (cần xác minh số điện thoại). Bấm **Save Version > Save & Run All (Commit)**: chạy nền tối đa 12 giờ, đóng trình duyệt vẫn chạy. Kết quả nằm ở tab *Output* (`/kaggle/working/deepweeds_lab`).
Chưa xong trong một lần commit thì: *Add Input > Your Work >* chọn chính notebook này (bản output trước) rồi commit lại. Ô "Nơi lưu kết quả" sẽ chép kết quả cũ vào, run đã xong không train lại.

**Colab:** `Runtime > Change runtime type > T4 GPU`, rồi *Run all*. Kết quả ghi vào Google Drive `MyDrive/deepweeds_lab/`.

- **Phiên bị ngắt thì cứ chạy lại từ đầu.** Run đã xong sẽ được đọc lại, không train lại; run đang dở sẽ chạy tiếp từ epoch cuối cùng đã lưu (`last.pt`).
- Mọi lựa chọn (backbone, kết hợp công thức, phương pháp suy luận) đều tính **trên val**. **Test chỉ chạy một lần cho mỗi seed** ở Bước 4; file test đã có thì code không chạy lại.
- Ước lượng trên T4 với 12 epoch: khoảng 25 lần huấn luyện, mỗi lần 8–20 phút tuỳ backbone, tổng khoảng 5–7 giờ GPU. Nên chia ra nhiều phiên.

## 0. Clone repo, cài thư viện, chạy test (CI)

In [ ]:
import os, sys, glob

IN_KAGGLE = os.path.exists("/kaggle/working")
LOCAL = "/kaggle/tmp" if IN_KAGGLE else "/content"   # đĩa cục bộ cho repo, dữ liệu, cache (không tính vào output)
os.makedirs(LOCAL, exist_ok=True)
REPO_URL = "https://github.com/Le-Anh-Duy/K4-Track4-Day2-Deeplearning-Advance.git"
BRANCH = "main"
REPO_DIR = f"{LOCAL}/K4-Track4-Day2-Deeplearning-Advance"
SUB_NAME = "2A202602723_le_anh_duy"         # phải trùng tên thư mục trong repo
# Repo private: tạo secret GH_TOKEN (Colab: biểu tượng chìa khoá; Kaggle: Add-ons > Secrets) rồi bỏ comment.
# if IN_KAGGLE:
#     from kaggle_secrets import UserSecretsClient; token = UserSecretsClient().get_secret("GH_TOKEN")
# else:
#     from google.colab import userdata; token = userdata.get("GH_TOKEN")
# REPO_URL = REPO_URL.replace("https://", f"https://{token}@")

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
SUB_DIR = f"{REPO_DIR}/submissions/{SUB_NAME}"
CODE_DIR = f"{SUB_DIR}/code"
%cd {CODE_DIR}
!git -C {REPO_DIR} log --oneline -1
!pip -q install timm openpyxl

In [ ]:
sys.path.insert(0, CODE_DIR)
import platform, math, copy, json, shutil
import numpy as np, pandas as pd, torch, timm, torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
assert torch.cuda.is_available(), "Chưa có GPU: Kaggle Settings > Accelerator, hoặc Colab Runtime > Change runtime type"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# CI (CPU, ~1 phút): test của repo gốc + test tự viết (focal γ=0, CutMix, gộp BN, ...) + chạy thử toàn bộ notebook
# trên dữ liệu giả với model timm test_* (không tải trọng số). Có lỗi thì sửa trước khi tốn giờ GPU.
!python -m unittest discover -s "{REPO_DIR}/tests" -q
!python -m unittest test_code -q
!python smoke_notebook.py 2>&1 | tail -1

In [ ]:
# Nơi lưu kết quả. Kaggle: /kaggle/working (thành Output khi commit; nếu có gắn output của lần commit trước
# làm Input thì chép vào để chạy tiếp). Colab: Google Drive (USE_DRIVE=False: lưu tạm trong /content).
if IN_KAGGLE:
    ART = "/kaggle/working/deepweeds_lab"
    prev = sorted(glob.glob("/kaggle/input/*/deepweeds_lab"))
    if prev and not os.path.exists(ART):
        print("Chép kết quả lần trước từ", prev[0]); shutil.copytree(prev[0], ART)
else:
    USE_DRIVE = True
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ART = "/content/drive/MyDrive/deepweeds_lab"
    else:
        ART = "/content/artifacts"
RUNS, PRED, CURVES, EVAL_OUT, FIGS = (f"{ART}/{d}" for d in ("runs", "predictions", "curves", "eval_out", "figures"))
for d in (RUNS, PRED, CURVES, EVAL_OUT, FIGS):
    os.makedirs(d, exist_ok=True)
print(ART)

### Tải dữ liệu
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả. Dữ liệu để trên đĩa cục bộ (`LOCAL/data`), không đọc ảnh từ Drive hay output.

In [ ]:
import hashlib

DATA = f"{LOCAL}/data"
LABELS_DIR = f"{DATA}/labels"
os.makedirs(LABELS_DIR, exist_ok=True)
if not os.path.exists(f"{DATA}/images.zip"):
    !wget -q -O {DATA}/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open(f"{DATA}/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"
!unzip -q -n {DATA}/images.zip -d {DATA}/images

BASE_URL = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    if not os.path.exists(f"{LABELS_DIR}/{name}.csv"):
        !wget -q -O {LABELS_DIR}/{name}.csv {BASE_URL}/{name}.csv

# Thư mục chứa nhiều .jpg nhất (zip có thể có hoặc không có thư mục con)
IMAGES_DIR = max((r for r, _, fs in os.walk(f"{DATA}/images")), key=lambda r: sum(f.endswith(".jpg") for f in os.listdir(r)))
print("IMAGES_DIR =", IMAGES_DIR, "|", sum(f.endswith(".jpg") for f in os.listdir(IMAGES_DIR)), "ảnh")

In [ ]:
import dataset, model as M, losses, inference, benchmark, train, experiments
from train import Config, run, load_config, load_model
from eval import compute_metrics
from dataset import CLASS_NAMES

train_df, val_df, test_df = dataset.load_split(LABELS_DIR)
split_info = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)   # S1-S4: dừng nếu vi phạm

# Giải mã toàn bộ ảnh một lần vào LOCAL/cache (~3,4 GB, vài phút); các lần chạy sau đọc thẳng từ đây.
CACHE = str(dataset.build_cache(IMAGES_DIR, pd.read_csv(f"{LABELS_DIR}/labels.csv")["Filename"],
                                f"{LOCAL}/cache/deepweeds_u8.npy"))
DEVICE = "cuda"
BASE = dict(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, cache=CACHE, out_dir=RUNS, pred_dir=PRED,
            curves_dir=CURVES, num_workers=min(4, os.cpu_count()), epochs=12)

def cfg(exp_id, desc, **kw):
    # Mọi thí nghiệm = công thức nền (Config mặc định, GUIDE mục 1.4) + đúng các thay đổi trong kw
    return Config(exp_id=exp_id, desc=desc, **{**BASE, **kw})

def val_row(probs, y=val_df["Label"].to_numpy()):
    m = compute_metrics(y, probs.argmax(1), probs)
    return {"macro_f1_val": m["macro_f1"], "top1_val": m["top1"], "ece_val": m["ece"],
            "f1_chinee_val": m["f1"][0], "f1_snake_val": m["f1"][7]}

## Bước 0 — EDA và kiểm tra pipeline

In [ ]:
PAPER_TABLE1 = [1125, 1064, 1031, 1022, 1062, 1009, 1074, 1016, 9106]
pc = split_info["per_class"].copy()
pc["paper_table1"] = PAPER_TABLE1
pc["lệch so với bài báo"] = pc["total"] - pc["paper_table1"]
pc["tỉ lệ (%)"] = (100 * pc["total"] / pc["total"].sum()).round(1)
display(pc)
print("Số ảnh:", split_info["n"], "| giao:", split_info["overlap"], "| hợp:", split_info["union"])

ax = pc[["train", "val", "test"]].plot.bar(stacked=True, figsize=(10, 4), title="Phân bố lớp DeepWeeds, fold 0")
ax.set_ylabel("số ảnh"); ax.set_xlabel("lớp"); plt.xticks(rotation=30, ha="right"); plt.tight_layout()
plt.savefig(f"{FIGS}/eda_class_distribution.png", dpi=130); plt.show()

In [ ]:
fig, axes = plt.subplots(3, 9, figsize=(20, 7))
for c in range(9):
    for r, f in enumerate(train_df[train_df.Label == c]["Filename"].sample(3, random_state=0)):
        axes[r, c].imshow(Image.open(f"{IMAGES_DIR}/{f}")); axes[r, c].axis("off")
    axes[0, c].set_title(CLASS_NAMES[c], fontsize=10)
plt.suptitle("Ảnh mẫu (3 ảnh mỗi lớp, tập train)"); plt.tight_layout()
plt.savefig(f"{FIGS}/eda_samples.png", dpi=110); plt.show()

In [ ]:
# Ảnh sau augmentation (đã giải chuẩn hoá) + nhãn, để chắc ảnh và nhãn khớp nhau; dòng cuối là CutMix.
MEAN, STD = torch.tensor(dataset.IMAGENET_MEAN)[:, None, None], torch.tensor(dataset.IMAGENET_STD)[:, None, None]
denorm = lambda x: (x * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()
sub = train_df.groupby("Label").head(1).head(6)
augs = ["basic", "color", "trivial"]
fig, axes = plt.subplots(len(augs) + 1, 6, figsize=(16, 11))
for r, a in enumerate(augs):
    ds = dataset.DeepWeedsDataset(sub, IMAGES_DIR, dataset.build_transforms(True, 224, a), CACHE)
    for c in range(6):
        x, y, _ = ds[c]; axes[r, c].imshow(denorm(x)); axes[r, c].set_title(f"{a}: {CLASS_NAMES[y]}", fontsize=9)
ds = dataset.DeepWeedsDataset(sub, IMAGES_DIR, dataset.build_transforms(True, 224, "basic"), CACHE)
xb = torch.stack([ds[i][0] for i in range(6)]); yb = torch.tensor([ds[i][1] for i in range(6)])
np.random.seed(0); torch.manual_seed(0)
xm, (ya, yb2, lam) = losses.mix_batch(xb, yb, 1.0, "cutmix")
for c in range(6):
    axes[-1, c].imshow(denorm(xm[c])); axes[-1, c].set_title(f"CutMix {CLASS_NAMES[ya[c]][:8]}/{CLASS_NAMES[yb2[c]][:8]} λ={lam:.2f}", fontsize=8)
for a in axes.flat: a.axis("off")
plt.tight_layout(); plt.savefig(f"{FIGS}/aug_preview.png", dpi=110); plt.show()

In [ ]:
# Kiểm tra pipeline: (1) loss ban đầu ≈ ln 9; (2) overfit được một batch nhỏ.
train.set_seed(0)
m = M.build_model("resnet50").to(DEVICE)
loader = dataset.make_loader(train_df, IMAGES_DIR, dataset.build_transforms(True, 224), 32, True, cache=CACHE, num_workers=2)
x, y, _ = next(iter(loader)); x, y = x.to(DEVICE), y.to(DEVICE)
m.eval()
with torch.no_grad():
    l0 = F.cross_entropy(m(x), y).item()
print(f"loss ban đầu = {l0:.3f}  (kỳ vọng ≈ ln 9 = {math.log(9):.3f})")

xs, ys = x[:16], y[:16]
opt = torch.optim.AdamW(m.parameters(), lr=3e-4)
m.train(); hist = []
for step in range(80):
    opt.zero_grad(); loss = F.cross_entropy(m(xs), ys); loss.backward(); opt.step(); hist.append(loss.item())
print(f"overfit 16 ảnh: loss {hist[0]:.3f} -> {hist[-1]:.4f}")
plt.figure(figsize=(5, 3)); plt.plot(hist); plt.yscale("log"); plt.xlabel("bước"); plt.ylabel("loss"); plt.title("Overfit 1 batch (16 ảnh)")
plt.tight_layout(); plt.savefig(f"{FIGS}/overfit_one_batch.png", dpi=120); plt.show()
del m, opt; torch.cuda.empty_cache()

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (AdamW, LR 1e-4 cho backbone / 1e-3 cho head, wd 0,05 không áp lên norm/bias, warmup 1 epoch + cosine, CE, AMP, 12 epoch, batch 64), cùng seed 0. Danh sách nằm ở `experiments.BACKBONES`.

In [ ]:
for exp, arch, short in experiments.BACKBONES:
    run(cfg(exp, short, backbone=arch))

In [ ]:
R = experiments.collect(RUNS)
B = R[R.exp_id.str.startswith("B")].sort_values("exp_id").reset_index(drop=True)
backbones = pd.DataFrame({
    "exp_id": B.exp_id, "backbone": B.backbone, "weights_tag": B.weights_tag, "params_M": B.params_m,
    "GMAC": B.gmacs, "img_size": B.img_size, "epochs": B.epochs, "seed": B.seed, "best_epoch": B.best_epoch,
    "macro_f1_val": B.val_macro_f1, "top1_val": B.val_top1, "f1_chinee_val": B["val_f1_Chinee Apple"],
    "f1_snake_val": B["val_f1_Snake Weed"], "train_time_per_epoch_s": B.train_time_per_epoch_s,
    "latency_b1_fp32_p50_ms": B.latency_b1_fp32_p50_ms, "gpu": B.gpu})
display(backbones)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, col, lab in ((ax[0], "latency_b1_fp32_p50_ms", "độ trễ p50 batch 1, FP32 (ms)"), (ax[1], "GMAC", "GMAC")):
    bk = backbones.dropna(subset=[col])
    a.scatter(bk[col], bk.macro_f1_val)
    for _, r in bk.iterrows():
        a.annotate(r.backbone, (r[col], r.macro_f1_val), fontsize=8)
    a.set_xlabel(lab); a.set_ylabel("macro-F1 val"); a.grid(alpha=.3)
plt.suptitle("Backbone: chất lượng và chi phí"); plt.tight_layout(); plt.savefig(f"{FIGS}/backbone_tradeoff.png", dpi=130); plt.show()

# Mặc định chọn backbone có macro-F1 val cao nhất. Đổi tay nếu muốn ưu tiên độ trễ, và GHI LÝ DO vào báo cáo.
BACKBONE = B.loc[B.val_macro_f1.idxmax(), "backbone"]
B_SRC = B.loc[B.backbone == BACKBONE, "exp_id"].iloc[0]
print("Backbone đi tiếp:", BACKBONE, f"({B_SRC})")

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi run khác `T00` đúng **một** yếu tố (`experiments.TRAINING`). `T00` chạy 3 seed để biết mức **nhiễu** (std); một chênh lệch Δ chỉ được coi là có thật khi |Δ| > std.
`T00` seed 0 trùng cấu hình với run backbone đã chọn, nên dùng lại run đó, không train lại.

In [ ]:
experiments.alias_run(load_config(RUNS, B_SRC, 0, **BASE), cfg("T00", "baseline", backbone=BACKBONE))
for seed in (1, 2):
    run(cfg("T00", "baseline", backbone=BACKBONE, seed=seed))
for exp, axis, what, desc, kw in experiments.TRAINING[1:]:
    run(cfg(exp, desc, backbone=BACKBONE, **kw))

In [ ]:
META = {e: dict(axis=a, what=w, desc=d, kw=kw) for e, a, w, d, kw in experiments.TRAINING}

def training_table():
    R = experiments.collect(RUNS)
    t00 = R[R.exp_id == "T00"]
    base0, noise = t00[t00.seed == 0].val_macro_f1.iloc[0], t00.val_macro_f1.std(ddof=1)
    T = R[R.exp_id.str.startswith("T") & (R.seed == 0)].sort_values("exp_id").reset_index(drop=True)
    tab = pd.DataFrame({
        "exp_id": T.exp_id, "backbone": T.backbone,
        "axis": T.exp_id.map(lambda e: META.get(e, {}).get("axis", "kết hợp")),
        "khác T00 ở": T.exp_id.map(lambda e: META.get(e, {}).get("what", COMBO_WHAT if e == "T11" else "")),
        "seed": T.seed, "macro_f1_val": T.val_macro_f1, "top1_val": T.val_top1,
        "delta_vs_T00": T.val_macro_f1 - base0, "f1_chinee_val": T["val_f1_Chinee Apple"],
        "f1_snake_val": T["val_f1_Snake Weed"], "best_epoch": T.best_epoch, "note": T.get("note")})
    tab["vượt nhiễu (|Δ| > std T00)"] = tab.delta_vs_T00.abs() > noise
    print(f"T00 macro-F1 val qua 3 seed: {t00.val_macro_f1.mean():.4f} ± {noise:.4f}")
    return tab, noise

COMBO_WHAT = ""
training, NOISE = training_table()
display(training)

In [ ]:
# T11 = kết hợp các yếu tố tốt (chọn trên VAL): mỗi trục lấy giá trị tốt nhất nếu Δ > std của T00.
# Nếu không trục nào vượt nhiễu thì lấy 2 trục có Δ dương lớn nhất (ghi rõ trong báo cáo). Có thể sửa COMBO bằng tay.
single = training[~training.exp_id.isin(["T00", "T11"])]
best_per_axis = single.loc[single.groupby("axis").delta_vs_T00.idxmax()].sort_values("delta_vs_T00", ascending=False)
picked = best_per_axis[best_per_axis.delta_vs_T00 > NOISE]
if picked.empty:
    picked = best_per_axis[best_per_axis.delta_vs_T00 > 0].head(2)
COMBO = {k: v for e in picked.exp_id for k, v in META[e]["kw"].items()}
COMBO_WHAT = " + ".join(META[e]["what"] for e in picked.exp_id)
print("T11 =", COMBO_WHAT, COMBO)
if COMBO:
    run(cfg("T11", "combo", backbone=BACKBONE, **COMBO))
training, NOISE = training_table()
display(training)

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**, dùng model tốt nhất ở Bước 2 (không train lại). Độ trễ: warmup 10 lần, `cuda.synchronize`, 100 lần đo, p50/p95/p99, **chỉ tính forward** (không tính tiền xử lý).

In [ ]:
R = experiments.collect(RUNS)
Tcand = R[R.exp_id.str.startswith("T") & (R.seed == 0)]
INF_EXP = Tcand.loc[Tcand.val_macro_f1.idxmax(), "exp_id"]
icfg = load_config(RUNS, INF_EXP, 0, **BASE)
imodel = load_model(icfg, DEVICE)
print("Model cho Bước 3:", INF_EXP, icfg.backbone)

ID = {"1view": "I00", "hflip": "I01", "5crop": "I02", "5crop_flip": "I02", "scale": "I02"}
NAME = {"1view": "1 view (resize 256, center crop 224)", "hflip": "TTA lật ngang", "5crop": "TTA 5 crop",
        "5crop_flip": "TTA 5 crop + lật", "scale": "TTA 3 tỉ lệ (224/256/288)"}
inf_rows, lat_rows, VIEW_LOGITS = [], [], {}

def add(exp, method, model_desc, k, probs, lat, **extra):
    inf_rows.append({"exp_id": exp, "method": method, "model": model_desc, "K": k, **val_row(probs),
                     "p50_ms": lat["p50"], "p95_ms": lat["p95"], "p99_ms": lat["p99"],
                     "images_per_s": lat["images_per_s"], **extra})

for method in experiments.VIEW_METHODS:
    try:
        _, yv, views = experiments.run_views(imodel, val_df, icfg, method, DEVICE)
        lat = benchmark.latency_report(imodel, 1, sizes=experiments.method_sizes(method))
    except Exception as e:  # ViT/Swin của timm cố định đầu vào 224: không chạy được đa tỉ lệ
        print(f"{method}: không chạy được với {icfg.backbone} ({type(e).__name__}); ghi vào báo cáo"); continue
    VIEW_LOGITS[method] = views
    for space in (("prob", "logit") if len(views) > 1 else ("prob",)):
        add(ID[method] if space == "prob" else "I03", f"{NAME[method]} · gộp {space}", INF_EXP, len(views),
            inference.aggregate_views(views, space), lat, method_key=method, space=space)
pd.DataFrame(inf_rows)

In [ ]:
# I04: dò độ phân giải kiểm tra (resize cả ảnh, FixRes). ViT/Swin của timm cố định đầu vào 224 nên sẽ báo lỗi và bỏ qua.
for s in (224, 256, 288, 320):
    try:
        _, _, v = experiments.run_views(imodel, val_df, icfg, f"res{s}", DEVICE)
    except Exception as e:
        print(f"res{s}: không chạy được với {icfg.backbone} ({type(e).__name__}); ghi vào báo cáo"); continue
    add("I04", f"độ phân giải {s} (cả ảnh)", INF_EXP, 1, inference.softmax(v[0]), benchmark.latency_report(imodel, 1, s),
        method_key=f"res{s}", space="prob")

# I05: ensemble 3 backbone tốt nhất ở Bước 1 (trung bình xác suất từ val_logits đã lưu)
top = B.nlargest(3, "val_macro_f1").exp_id.tolist()
probs = inference.ensemble_probs([inference.softmax(np.load(f"{RUNS}/{e}/seed0/val_logits.npy")) for e in top])
ens_models = [load_model(load_config(RUNS, e, 0, **BASE), DEVICE) for e in top]
add("I05", "ensemble 3 backbone", "+".join(top), 3, probs, benchmark.latency_report(ens_models, 1, 224))
del ens_models; torch.cuda.empty_cache()

# I06: trọng số EMA (T10) so với trọng số thường cùng run, không tốn thêm khi suy luận
t10 = R[(R.exp_id == "T10") & (R.seed == 0)].iloc[0]
lat_t10 = benchmark.latency_report(load_model(load_config(RUNS, "T10", 0, **BASE), DEVICE), 1, 224)
add("I06", "trọng số EMA", "T10", 1, inference.softmax(np.load(f"{RUNS}/T10/seed0/val_logits.npy")), lat_t10)
inf_rows.append({"exp_id": "I06", "method": "trọng số thường (không EMA), cùng epoch", "model": "T10", "K": 1,
                 "macro_f1_val": t10.val_macro_f1_raw_at_best, "p50_ms": lat_t10["p50"]})

# I07: temperature scaling. T khớp trên VAL; ECE trước/sau ở đây đo trên chính val (lạc quan), ECE test xem ở Bước 4.
z = VIEW_LOGITS["1view"][0]
T_val = inference.fit_temperature(z, val_df["Label"].to_numpy())
lat1 = benchmark.latency_report(imodel, 1, 224)
add("I07", f"temperature scaling T={T_val:.3f}", INF_EXP, 1, inference.apply_temperature(z, T_val), lat1, T=T_val,
    ece_val_before=val_row(inference.softmax(z))["ece_val"])
pd.DataFrame(inf_rows).tail(8)

In [ ]:
# I08: gộp BatchNorm + FP16/AMP. Cần mạng có BN: dùng model Bước 3 nếu có BN, nếu không thì dùng B01 (ResNet-50).
has_bn = any(isinstance(mm, torch.nn.BatchNorm2d) for mm in imodel.modules())
bn_exp = INF_EXP if has_bn else "B01"
bcfg = load_config(RUNS, bn_exp, 0, **BASE); bmodel = load_model(bcfg, DEVICE)
fused = inference.fuse_conv_bn(bmodel)
vloader = dataset.make_loader(val_df, IMAGES_DIR, dataset.build_transforms(False, 224), 128, False, cache=CACHE, num_workers=BASE["num_workers"])
_, _, lo = inference.predict_logits(bmodel, vloader, DEVICE, amp=False)
_, _, lf = inference.predict_logits(fused, vloader, DEVICE, amp=False)
print(f"{bn_exp}: gộp {fused.fused_pairs} cặp conv+BN, sai số logit lớn nhất {np.abs(lo - lf).max():.2e}")
half = copy.deepcopy(fused).half()
_, _, lh = inference.predict_logits(half, vloader, DEVICE, amp=False)
del half
for desc, logits, mdl, dt, fb in (("FP32", lo, bmodel, "fp32", False), ("gộp BN, FP32", lf, fused, "fp32", True),
                                  ("AMP (autocast)", lo, bmodel, "amp", False), ("gộp BN + FP16", lh, fused, "fp16", True)):
    add("I08", desc, bn_exp, 1, inference.softmax(logits), benchmark.latency_report(mdl, 1, 224, dtype=dt, fused_bn=fb))

# Sheet Latency: batch 1 (robot) và batch 32 (thông lượng)
light = B.nsmallest(1, "gmacs").exp_id.iloc[0]
lmodel = load_model(load_config(RUNS, light, 0, **BASE), DEVICE)
for name, mdl, fb in ((f"{INF_EXP} {icfg.backbone}", imodel, False), (f"{bn_exp} {bcfg.backbone}", bmodel, False),
                      (f"{bn_exp} {bcfg.backbone} (gộp BN)", fused, True), (f"{light} (mạng nhẹ nhất theo GMAC)", lmodel, False)):
    for bs in (1, 32):
        for dt in ("fp32", "amp", "fp16"):
            lat_rows.append({"config": name, **benchmark.latency_report(mdl, bs, 224, dtype=dt, fused_bn=fb)})
del lmodel; torch.cuda.empty_cache()
lat_df = pd.DataFrame(lat_rows)
inf_df = pd.DataFrame(inf_rows)
i00 = inf_df.loc[inf_df.exp_id == "I00", "p50_ms"].iloc[0]
inf_df["relative_cost_vs_I00"] = inf_df.p50_ms / i00
inf_df.to_csv(f"{ART}/inference_val.csv", index=False); lat_df.to_csv(f"{ART}/latency.csv", index=False)
display(inf_df); display(lat_df)

In [ ]:
ok = inf_df.dropna(subset=["p50_ms", "macro_f1_val"])
plt.figure(figsize=(9, 5))
plt.scatter(ok.p50_ms, ok.macro_f1_val)
for _, r in ok.iterrows():
    plt.annotate(f"{r.exp_id} {r.method[:22]}", (r.p50_ms, r.macro_f1_val), fontsize=7)
plt.axvline(100, color="r", ls=":", label="ngân sách 100 ms")
plt.xscale("log"); plt.xlabel("độ trễ p50, batch 1 (ms, thang log)"); plt.ylabel("macro-F1 val")
plt.title("Đánh đổi độ chính xác và độ trễ (val)"); plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(f"{FIGS}/inference_tradeoff.png", dpi=130); plt.show()

## Bước 4 — Chung kết: 3 seed, test một lần mỗi seed
- **F01** = công thức của `INF_EXP` (tốt nhất trên val ở Bước 2) + phương pháp suy luận một model tốt nhất trên val (I00–I03) + temperature scaling (T khớp trên val).
- **Mốc** = `T00` + `I00` (1 view), cùng 3 seed.
- F01 seed 0 trùng cấu hình `INF_EXP` seed 0 nên dùng lại run đó.

In [ ]:
cand = inf_df[inf_df.exp_id.isin(["I00", "I01", "I02", "I03"])]
best = cand.loc[cand.macro_f1_val.idxmax()]
FINAL_METHOD, FINAL_SPACE = best.method_key, best.space
FINAL_KW = {k: getattr(icfg, k) for k in Config.__dataclass_fields__ if k not in BASE and k not in ("exp_id", "desc", "seed")}
print("F01 =", INF_EXP, FINAL_KW, "| suy luận:", FINAL_METHOD, FINAL_SPACE, "+ temperature scaling")

experiments.alias_run(icfg, cfg("F01", "final", **FINAL_KW))
for seed in (0, 1, 2):
    c = cfg("F01", "final", seed=seed, **FINAL_KW)
    run(c)
    print(experiments.final_predict(c, FINAL_METHOD, FINAL_SPACE, temperature=True, device=DEVICE))
for seed in (0, 1, 2):
    print(experiments.final_predict(load_config(RUNS, "T00", seed, **BASE), "1view", device=DEVICE))

In [ ]:
# Cấu hình thời gian thực cho I5: p95 batch 1 của F01 với phương pháp suy luận đã chọn (FP32)
fmodel = load_model(load_config(RUNS, "F01", 0, **BASE), DEVICE)
rt = benchmark.latency_report(fmodel, 1, sizes=experiments.method_sizes(FINAL_METHOD))
REALTIME_P95 = f"{rt['p95']:.2f}"
print(f"F01 ({FINAL_METHOD}) batch 1 FP32: p50 {rt['p50']:.1f} ms, p95 {rt['p95']:.1f} ms trên {rt['gpu']}")
del fmodel; torch.cuda.empty_cache()

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (eval.py gốc, không sửa)
!python "{REPO_DIR}/eval.py" score --pred "{PRED}/F01_seed*_test.csv" --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out {EVAL_OUT}
!python "{REPO_DIR}/eval.py" score --pred "{PRED}/T00_seed*_test.csv" --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out {EVAL_OUT}

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất)
!python "{REPO_DIR}/eval.py" grade --final "{PRED}/F01_seed*_test.csv" --baseline "{PRED}/T00_seed*_test.csv" --uncal "{PRED}/F01uncal_seed*_test.csv" --final-val "{PRED}/F01_seed*_val.csv" --val-csv {LABELS_DIR}/val_subset0.csv --latency-p95-ms {REALTIME_P95} --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out {EVAL_OUT}

In [ ]:
# Ma trận nhầm lẫn (tổng 3 seed, chuẩn hoá theo hàng) + ảnh nhầm giữa Chinee Apple và Snake Weed
cm = pd.read_csv(f"{EVAL_OUT}/F01_confusion_sum.csv", index_col=0).to_numpy()
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cmn, cmap="Blues")
for i in range(9):
    for j in range(9):
        ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white" if cmn[i, j] > .5 else "black")
ax.set_xticks(range(9), CLASS_NAMES, rotation=45, ha="right"); ax.set_yticks(range(9), CLASS_NAMES)
ax.set_xlabel("dự đoán"); ax.set_ylabel("nhãn thật"); ax.set_title("F01, test, tổng 3 seed (chuẩn hoá theo hàng)")
plt.colorbar(im); plt.tight_layout(); plt.savefig(f"{FIGS}/confusion_F01_test.png", dpi=130); plt.show()

p = pd.read_csv(f"{PRED}/F01_seed0_test.csv")
wrong = p[(p.y_true != p.y_pred) & p.y_true.isin([0, 7]) & p.y_pred.isin([0, 7])].head(12)
if len(wrong):
    fig, axes = plt.subplots(2, 6, figsize=(16, 6))
    for a, (_, r) in zip(axes.flat, wrong.iterrows()):
        a.imshow(Image.open(f"{IMAGES_DIR}/{r.Filename}"))
        a.set_title(f"thật {CLASS_NAMES[r.y_true][:11]}\nđoán {CLASS_NAMES[r.y_pred][:11]} ({r[f'p{r.y_pred}']:.2f})", fontsize=8)
    for a in axes.flat: a.axis("off")
    plt.tight_layout(); plt.savefig(f"{FIGS}/errors_chinee_snake.png", dpi=110); plt.show()

## Bước 5 — `results.xlsx` và chép sản phẩm vào thư mục bài nộp

In [ ]:
final, per_class = experiments.final_tables(PRED, ["F01", "T00"], f"{LABELS_DIR}/test_subset0.csv")
summary = pd.concat([
    backbones.assign(kind="backbone", desc=backbones.backbone)[["kind", "exp_id", "desc", "macro_f1_val", "top1_val", "latency_b1_fp32_p50_ms"]],
    training.assign(kind="training", desc=training["khác T00 ở"],
                    latency_b1_fp32_p50_ms=np.nan)[["kind", "exp_id", "desc", "macro_f1_val", "top1_val", "latency_b1_fp32_p50_ms"]],
    inf_df.assign(kind="inference", desc=inf_df.method + " · " + inf_df.model,
                  latency_b1_fp32_p50_ms=inf_df.p50_ms)[["kind", "exp_id", "desc", "macro_f1_val", "top1_val", "latency_b1_fp32_p50_ms"]],
]).sort_values("macro_f1_val", ascending=False).head(10).reset_index(drop=True)
fin = final[final.seed.astype(str).str.startswith("mean")]
summary = pd.concat([summary, pd.DataFrame([{"kind": "TEST chung kết", "exp_id": r.exp_id, "desc": f"macro-F1 test {r.macro_f1_test} | top-1 {r.top1_test}"} for r in fin.itertuples()])], ignore_index=True)

xlsx = experiments.write_results_xlsx(f"{ART}/results.xlsx", {
    "Summary": summary, "Backbones": backbones, "Training": training, "Inference": inf_df,
    "Final": final, "PerClass": per_class, "Latency": lat_df},
    highlight={"Backbones": "macro_f1_val", "Training": "macro_f1_val", "Inference": "macro_f1_val", "Summary": "macro_f1_val"})
print(xlsx); display(summary)

In [ ]:
# Chép sản phẩm nhỏ vào submissions/<tên>/ (không chép checkpoint), rồi nén để tải về và commit từ máy.
import glob
os.makedirs(f"{SUB_DIR}/predictions", exist_ok=True)
for f in glob.glob(f"{PRED}/F01*") + glob.glob(f"{PRED}/T00_seed*_test.csv"):
    shutil.copy(f, f"{SUB_DIR}/predictions/")
for src, dst in ((CURVES, "curves"), (FIGS, "figures"), (EVAL_OUT, "eval_out_copy")):
    shutil.copytree(src, f"{SUB_DIR}/{dst}", dirs_exist_ok=True)
shutil.copy(f"{ART}/results.xlsx", SUB_DIR)
!pip freeze | grep -iE "^(torch|torchvision|timm|numpy|pandas|scipy|openpyxl|matplotlib)==" > {SUB_DIR}/requirements-colab.txt
if IN_KAGGLE:  # tải ở tab Output của version đã commit
    shutil.make_archive("/kaggle/working/submission", "zip", SUB_DIR)
else:
    shutil.make_archive("/content/submission", "zip", SUB_DIR)
    from google.colab import files
    files.download("/content/submission.zip")